In [2]:

from pathlib import Path
import pandas as pd

# Tìm thư mục dự án
root = Path.cwd().resolve()
PROJECT_DIR = next(
    p for p in [root, *root.parents]
    if (p / "results/related_methods/model_baselines/evaluation/baseline_evaluation_detail.csv").exists()
)

BASELINE_DIR = PROJECT_DIR / "results/related_methods/model_baselines"
OUTPUT_DIR = PROJECT_DIR / "results/related_methods/ontology_guided_repair"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = 1

# Đọc kết quả đánh giá của 4 mô hình
baseline_eval = pd.read_csv(
    BASELINE_DIR / "evaluation/baseline_evaluation_detail.csv",
    keep_default_na=False
)
baseline_eval = baseline_eval[baseline_eval["Run"] == RUN_ID].copy()

# Đọc SQL đã sinh từ các mô hình
models = {
    "Qwen2.5-Coder-3B": "qwen2_5_coder_3b",
    "NSQL-350M": "nsql_350m",
    "SQLCoder-7B-2": "sqlcoder_7b_2",
    "NSQL-Llama-2-7B": "nsql_llama_2_7b"
}

frames = []

for name, slug in models.items():
    path = BASELINE_DIR / f"{slug}_run_{RUN_ID:02d}.csv"
    df = pd.read_csv(path, keep_default_na=False)
    df = df[["ID", "Generated Query", "Status"]].copy()
    df["Model"] = name
    frames.append(df)

generated_sql = pd.concat(frames, ignore_index=True)

# Ghép SQL đã sinh với kết quả đánh giá
repair_data = baseline_eval.merge(
    generated_sql,
    on=["Model", "ID"],
    validate="one_to_one"
)

assert len(repair_data) == 172, "Thiếu dữ liệu baseline"

# Thống kê kết quả ban đầu
summary = repair_data.groupby("Model").agg(
    Questions=("ID", "count"),
    Correct=("Correct", "sum")
)

summary["EA (%)"] = (summary["Correct"] / 43 * 100).round(2)

print("Project:", PROJECT_DIR)
print("Số mô hình:", len(models))
print("Số câu SQL:", len(repair_data))
print("Thư mục kết quả:", OUTPUT_DIR)

display(summary)

Project: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Số mô hình: 4
Số câu SQL: 172
Thư mục kết quả: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\related_methods\ontology_guided_repair


,Questions,Correct,EA (%)
Model,,,
NSQL-350M,43,3,6.98
NSQL-Llama-2-7B,43,0,0.00
Qwen2.5-Coder-3B,43,4,9.30
SQLCoder-7B-2,43,4,9.30


In [3]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, OWL, RDFS

# Tìm các file RDF trong dự án
folders = [
    PROJECT_DIR / "data",
    PROJECT_DIR / "results/ontop",
    PROJECT_DIR / "results/ontology",
    PROJECT_DIR / "ontology"
]

files = sorted(set(
    p for folder in folders if folder.exists()
    for p in folder.rglob("*")
    if p.suffix.lower() in {".ttl", ".owl", ".rdf"}
))

RR = Namespace("http://www.w3.org/ns/r2rml#")

ontologies = []
mappings = []

# Phân loại theo nội dung RDF
for path in files:
    try:
        graph = Graph()
        graph.parse(path)
    except Exception:
        continue

    n_classes = len(set(graph.subjects(RDF.type, OWL.Class))) + len(
        set(graph.subjects(RDF.type, RDFS.Class))
    )
    n_maps = len(set(graph.subjects(RDF.type, RR.TriplesMap)))

    if n_classes:
        ontologies.append((path, graph))
    if n_maps:
        mappings.append((path, graph))

print("Ontology:", [str(p) for p, _ in ontologies])
print("R2RML:", [str(p) for p, _ in mappings])

assert len(ontologies) == 1, "Cần xác định đúng file ontology"
assert len(mappings) == 1, "Cần xác định đúng file mapping"

ontology_file, ontology = ontologies[0]
mapping_file, mapping = mappings[0]

print("Ontology triples:", len(ontology))
print("Mapping triples:", len(mapping))
print("R2RML Triples Maps:", len(set(
    mapping.subjects(RDF.type, RR.TriplesMap)
)))


Ontology: ['C:\\Users\\Win 11 New\\Desktop\\thuc-nghiem-co-so-httt\\data\\ACME_Insurance\\ontology\\insurance.ttl']
R2RML: ['C:\\Users\\Win 11 New\\Desktop\\thuc-nghiem-co-so-httt\\results\\ontop\\acme_mapping.ttl']
Ontology triples: 164
Mapping triples: 561
R2RML Triples Maps: 62


In [4]:
import re
from rdflib.namespace import RDF, RDFS, OWL

RR = Namespace("http://www.w3.org/ns/r2rml#")

# Đọc DDL và câu hỏi gốc
ddl = (PROJECT_DIR / "data/ACME_Insurance/DDL/ACME_small.ddl").read_text(encoding="utf-8-sig")
questions = pd.read_csv(
    PROJECT_DIR / "results/llm/reproduction/generation_run_01.csv",
    keep_default_na=False
)[["ID", "Question"]].drop_duplicates("ID")

# Rút gọn tên các thành phần RDF
def short_name(value):
    return str(value).split("#")[-1].rstrip("/").split("/")[-1]

# Trích xuất thông tin ngữ nghĩa từ ontology
relations = {RDFS.label, RDFS.domain, RDFS.range, RDFS.subClassOf}
types = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty}

ontology_info = sorted({
    f"{short_name(s)} {short_name(p)} {short_name(o)}"
    for s, p, o in ontology
    if p in relations or (p == RDF.type and o in types)
})

# Trích xuất ánh xạ giữa ontology và các bảng SQL
mapping_info = []

for tm in mapping.subjects(RDF.type, RR.TriplesMap):
    logical = mapping.value(tm, RR.logicalTable)
    table = mapping.value(logical, RR.tableName) if logical else None
    sql = mapping.value(logical, RR.sqlQuery) if logical else None

    sources = re.findall(r"\b(?:FROM|JOIN)\s+([\w.]+)", str(sql or ""), re.I)
    source = str(table) if table else ", ".join(dict.fromkeys(sources))

    for pom in mapping.objects(tm, RR.predicateObjectMap):
        predicate = mapping.value(pom, RR.predicate)
        pm = mapping.value(pom, RR.predicateMap)
        if predicate is None and pm:
            predicate = mapping.value(pm, RR.constant)

        om = mapping.value(pom, RR.objectMap)
        target = None
        if om:
            target = (
                mapping.value(om, RR.column)
                or mapping.value(om, RR.template)
                or mapping.value(om, RR.parentTriplesMap)
            )

        if predicate:
            mapping_info.append(
                f"{source}: {short_name(predicate)} -> {short_name(target) if target else '?'}"
            )

semantic_context = (
    "Ontology:\n" + "\n".join(ontology_info)
    + "\nR2RML mappings:\n" + "\n".join(sorted(set(mapping_info)))
)

# Chỉ chọn các SQL bị lỗi thực thi
failed = repair_data[
    repair_data["Execution Status"] == "ERROR"
].merge(questions, on="ID", validate="many_to_one")

# Tạo prompt cho hai phương pháp sửa SQL
prompts = []

for _, row in failed.iterrows():
    for method in ["schema_only", "ontology_guided"]:
        context = ddl
        if method == "ontology_guided":
            context += "\n" + semantic_context

        prompt = (
            "Repair the SQL query using the database information below.\n"
            "Use only existing tables and columns. Return one read-only SELECT query.\n"
            "Do not explain or include Markdown.\n\n"
            f"Database information:\n{context}\n\n"
            f"Question: {row['Question']}\n"
            f"Original SQL: {row['Generated Query']}\n"
            f"DuckDB error: {row['Error']}\n"
            "Corrected SQL:"
        )

        prompts.append({
            "Model": row["Model"],
            "ID": row["ID"],
            "Method": method,
            "Prompt": prompt
        })

repair_prompts = pd.DataFrame(prompts)

print("Ontology facts:", len(ontology_info))
print("Mapping relations:", len(set(mapping_info)))
print("SQL cần sửa:", len(failed))
print("Số repair prompts:", len(repair_prompts))

display(repair_prompts.groupby(["Model", "Method"]).size().unstack(fill_value=0))

Ontology facts: 123
Mapping relations: 50
SQL cần sửa: 142
Số repair prompts: 284


Method,ontology_guided,schema_only
Model,,
NSQL-350M,38,38
NSQL-Llama-2-7B,41,41
Qwen2.5-Coder-3B,32,32
SQLCoder-7B-2,31,31


In [5]:
import json
import time
import hashlib
import urllib.request
import pandas as pd

# Cấu hình mô hình sửa SQL
MODEL_ID = "qwen2.5-coder:3b-instruct"
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
RESULT_FILE = OUTPUT_DIR / "repair_predictions_run_01.csv"

# Kiểm tra Ollama và mô hình
with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=15) as response:
    installed = [m["name"] for m in json.load(response)["models"]]

assert MODEL_ID in installed, f"Chưa tải mô hình {MODEL_ID}"

# Gọi mô hình để sửa câu SQL
def repair_sql(prompt):
    data = {
        "model": MODEL_ID,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "seed": 7,
            "num_ctx": 8192,
            "num_predict": 512
        }
    }

    request = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(data).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with urllib.request.urlopen(request, timeout=900) as response:
        result = json.load(response)

    return result.get("response", "").strip()

# Đọc checkpoint nếu đã chạy trước đó
cache = {}

if RESULT_FILE.exists():
    previous = pd.read_csv(RESULT_FILE, keep_default_na=False)
    for _, row in previous.iterrows():
        key = (row["Model"], int(row["ID"]), row["Method"])
        cache[key] = row.to_dict()

# Chạy toàn bộ prompts, lưu sau từng câu
results = []

assert not repair_prompts.duplicated(["Model", "ID", "Method"]).any()

for i, (_, row) in enumerate(repair_prompts.iterrows(), 1):
    key = (row["Model"], int(row["ID"]), row["Method"])
    prompt_hash = hashlib.sha256(row["Prompt"].encode("utf-8")).hexdigest()
    old = cache.get(key)

    if (old and old.get("Status") == "GENERATED"
            and old.get("Prompt Hash") == prompt_hash
            and str(old.get("Repaired SQL", "")).strip()):
        results.append(old)
        print(f"{i}/{len(repair_prompts)}: Reused")
        continue

    start = time.perf_counter()

    try:
        sql = repair_sql(row["Prompt"])
        status = "GENERATED" if sql else "EMPTY"
        error = ""
    except Exception as e:
        sql = ""
        status = "ERROR"
        error = str(e)

    results.append({
        "Model": row["Model"],
        "ID": row["ID"],
        "Method": row["Method"],
        "Repair Model": MODEL_ID,
        "Prompt Hash": prompt_hash,
        "Repaired SQL": sql,
        "Status": status,
        "Error": error,
        "Time (s)": round(time.perf_counter() - start, 3)
    })

    # Lưu checkpoint an toàn sau mỗi câu
    temp_file = RESULT_FILE.with_suffix(".tmp")
    pd.DataFrame(results).to_csv(temp_file, index=False, encoding="utf-8-sig")
    temp_file.replace(RESULT_FILE)

    print(f"{i}/{len(repair_prompts)}: {row['Model']} Q{int(row['ID']):02d} "
          f"{row['Method']} - {status}", flush=True)

    if status == "ERROR":
        raise RuntimeError(f"Inference bị gián đoạn: {error}")

repair_results = pd.DataFrame(results)

print("\nTổng số prompts:", len(repair_prompts))
print("Đã sinh:", (repair_results["Status"] == "GENERATED").sum())
print("Kết quả:", RESULT_FILE)

display(repair_results.groupby(["Model", "Method", "Status"]).size())

1/284: Reused
2/284: Reused
3/284: Reused
4/284: Reused
5/284: Reused
6/284: Reused
7/284: Reused
8/284: Reused
9/284: Reused
10/284: Reused
11/284: Reused
12/284: Reused
13/284: Reused
14/284: Reused
15/284: Reused
16/284: Reused
17/284: Reused
18/284: Reused
19/284: Reused
20/284: Reused
21/284: Reused
22/284: Reused
23/284: Reused
24/284: Reused
25/284: Reused
26/284: Reused
27/284: Reused
28/284: Reused
29/284: Reused
30/284: Reused
31/284: Reused
32/284: Reused
33/284: Reused
34/284: Reused
35/284: Reused
36/284: Reused
37/284: Reused
38/284: Reused
39/284: Reused
40/284: Reused
41/284: Reused
42/284: Reused
43/284: Reused
44/284: Reused
45/284: Reused
46/284: Reused
47/284: Reused
48/284: Reused
49/284: Reused
50/284: Reused
51/284: Reused
52/284: Reused
53/284: Reused
54/284: Reused
55/284: Reused
56/284: Reused
57/284: Reused
58/284: Reused
59/284: Reused
60/284: Reused
61/284: Reused
62/284: Reused
63/284: Reused
64/284: Reused
65/284: Reused
66/284: Reused
67/284: Reused
68/2

Model             Method           Status   
NSQL-350M         ontology_guided  GENERATED    38
                  schema_only      GENERATED    38
NSQL-Llama-2-7B   ontology_guided  GENERATED    41
                  schema_only      GENERATED    41
Qwen2.5-Coder-3B  ontology_guided  GENERATED    32
                  schema_only      GENERATED    32
SQLCoder-7B-2     ontology_guided  GENERATED    31
                  schema_only      GENERATED    31
dtype: int64

In [7]:
import ast
import json
import re
import time
import duckdb
from collections import Counter
from decimal import Decimal

# Đọc hàm đánh giá từ notebook gốc
eval_file = PROJECT_DIR / "notebook/05_evaluation.ipynb"
notebook = json.loads(eval_file.read_text(encoding="utf-8"))

functions = {}

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue
    try:
        tree = ast.parse("".join(cell["source"]))
    except SyntaxError:
        continue
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in {"normalize", "same_result"}:
            functions[node.name] = node

assert len(functions) == 2, "Không tìm thấy hàm đánh giá gốc"

module = ast.fix_missing_locations(
    ast.Module(body=list(functions.values()), type_ignores=[])
)
env = {"pd": pd, "re": re, "Decimal": Decimal, "Counter": Counter}
exec(compile(module, "<evaluation>", "exec"), env)
same_result = env["same_result"]

# Đọc kết quả SQL chuẩn
ref_dir = PROJECT_DIR / "results/sql/reference_results"
ref_map = pd.read_csv(
    PROJECT_DIR / "results/ontop/reference_comparison.csv",
    keep_default_na=False
)
ref_map["ID"] = ref_map["ID"].astype(int)

references = {
    int(row["ID"]): ref_dir / (str(row["SQL Reference"]).split("/")[-1] + ".csv")
    for _, row in ref_map.iterrows()
}

# Chuẩn hóa định dạng đầu ra, không sửa nội dung SQL
def clean_sql(text):
    text = str(text).strip()
    match = re.fullmatch(r"```(?:sql)?\s*\n(.*?)\n\s*```", text, re.I | re.S)
    if match:
        text = match.group(1).strip()
    if text.startswith("[SQL]"):
        text = text[5:].strip()
    if "[/SQL]" in text:
        text = text.split("[/SQL]")[0].strip()
    return text

# Đánh giá toàn bộ SQL sau repair
eval_dir = OUTPUT_DIR / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

assert not repair_results.duplicated(["Model", "ID", "Method"]).any()
assert len(repair_results) == len(repair_prompts)

records = []
con = duckdb.connect(
    str(PROJECT_DIR / "results/sql/acme.duckdb"),
    read_only=True
)

try:
    for _, row in repair_results.iterrows():
        qid = int(row["ID"])
        sql = clean_sql(row["Repaired SQL"])
        status, error, correct = "SKIPPED", "", 0
        start = time.perf_counter()

        folder = eval_dir / "query_results" / row["Method"] / models[row["Model"]]
        folder.mkdir(parents=True, exist_ok=True)
        output = folder / f"{qid:02d}.csv"
        output.unlink(missing_ok=True)

        if row["Status"] == "GENERATED":
            try:
                if not re.match(r"^(SELECT|WITH)\b", sql, re.I):
                    raise ValueError("Not a read-only SELECT/WITH query")
                if ";" in sql.rstrip().rstrip(";"):
                    raise ValueError("Multiple statements not permitted")
                if re.search(
                    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|ATTACH|COPY|EXPORT|INSTALL|LOAD)\b",
                    sql, re.I
                ):
                    raise ValueError("Not a read-only SELECT/WITH query")

                con.execute(sql).fetchdf().to_csv(output, index=False)
                correct = int(same_result(references[qid], output))
                status = "OK"

            except Exception as e:
                status, error = "ERROR", f"{type(e).__name__}: {e}"
                output.unlink(missing_ok=True)

        records.append({
            "Model": row["Model"],
            "ID": qid,
            "Method": row["Method"],
            "Repaired SQL": sql,
            "Execution Status": status,
            "Correct": correct,
            "Error": error,
            "Time (s)": round(time.perf_counter() - start, 3)
        })
finally:
    con.close()

repair_eval = pd.DataFrame(records)
repair_eval.to_csv(
    eval_dir / "repair_evaluation_detail.csv",
    index=False, encoding="utf-8-sig"
)

# So sánh với kết quả baseline trên toàn bộ 43 câu/model
baseline_score = baseline_eval.groupby("Model")["Correct"].sum()

comparison = repair_eval.groupby(["Model", "Method"]).agg(
    Evaluated=("ID", "count"),
    Executed=("Execution Status", lambda x: (x == "OK").sum()),
    Recovered=("Correct", "sum")
).reset_index()

comparison["Baseline Correct"] = comparison["Model"].map(baseline_score)
comparison["Correct After"] = comparison["Baseline Correct"] + comparison["Recovered"]
comparison["EA Before (%)"] = (comparison["Baseline Correct"] / 43 * 100).round(2)
comparison["EA After (%)"] = (comparison["Correct After"] / 43 * 100).round(2)
comparison["Delta EA (pp)"] = (
    comparison["EA After (%)"] - comparison["EA Before (%)"]
).round(2)

display(comparison)
print("Đã lưu:", eval_dir / "repair_evaluation_detail.csv")


,Model,Method,Evaluated,Executed,Recovered,Baseline Correct,Correct After,EA Before (%),EA After (%),Delta EA (pp)
0,NSQL-350M,ontology_guided,38,5,0,3,3,6.98,6.98,0.00
1,NSQL-350M,schema_only,38,4,0,3,3,6.98,6.98,0.00
2,NSQL-Llama-2-7B,ontology_guided,41,4,3,0,3,0.00,6.98,6.98
3,NSQL-Llama-2-7B,schema_only,41,8,4,0,4,0.00,9.30,9.30
4,Qwen2.5-Coder-3B,ontology_guided,32,3,0,4,4,9.30,9.30,0.00
5,Qwen2.5-Coder-3B,schema_only,32,3,0,4,4,9.30,9.30,0.00
6,SQLCoder-7B-2,ontology_guided,31,4,0,4,4,9.30,9.30,0.00
7,SQLCoder-7B-2,schema_only,31,3,0,4,4,9.30,9.30,0.00


Đã lưu: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\related_methods\ontology_guided_repair\evaluation\repair_evaluation_detail.csv


In [8]:
# Tạo bảng đánh giá đầy đủ cho cả hai phương pháp
base = baseline_eval[
    ["Model", "ID", "Correct", "Execution Status"]
].copy()

base = base.rename(columns={
    "Correct": "Baseline Correct",
    "Execution Status": "Baseline Status"
})
base["ID"] = base["ID"].astype(int)

methods = ["schema_only", "ontology_guided"]
full = pd.concat(
    [base.assign(Method=method) for method in methods],
    ignore_index=True
)

# Ghép kết quả sửa SQL
repaired = repair_eval.rename(columns={
    "Correct": "Repair Correct",
    "Execution Status": "Repair Status"
})

full = full.merge(
    repaired[["Model", "ID", "Method", "Repair Correct", "Repair Status"]],
    on=["Model", "ID", "Method"],
    how="left",
    validate="one_to_one"
)

# Giữ kết quả gốc đối với các câu không cần sửa
full["Correct After"] = full["Repair Correct"].fillna(
    full["Baseline Correct"]
).astype(int)

full["Final Status"] = full["Repair Status"].fillna(
    full["Baseline Status"]
)

# Kiểm tra tính đầy đủ
assert len(full) == 43 * 4 * 2
assert not full.duplicated(["Model", "ID", "Method"]).any()

needs_repair = full["Baseline Status"].eq("ERROR")
assert full.loc[needs_repair, "Repair Status"].notna().all()
assert full.loc[~needs_repair, "Repair Status"].isna().all()

# Tổng hợp Execution Accuracy
final_summary = full.groupby(["Model", "Method"]).agg(
    Questions=("ID", "count"),
    Baseline_Correct=("Baseline Correct", "sum"),
    Correct_After=("Correct After", "sum")
).reset_index()

final_summary["EA Before (%)"] = (
    final_summary["Baseline_Correct"] / 43 * 100
).round(2)

final_summary["EA After (%)"] = (
    final_summary["Correct_After"] / 43 * 100
).round(2)

final_summary["Delta EA (pp)"] = (
    final_summary["EA After (%)"] - final_summary["EA Before (%)"]
).round(2)

# Thống kê trạng thái thực thi sau sửa
error_summary = (
    repair_eval.groupby(["Model", "Method", "Execution Status"])
    .size()
    .reset_index(name="Count")
)

# Lưu các bảng kết quả
eval_dir = OUTPUT_DIR / "evaluation"

full.to_csv(eval_dir / "repair_full_detail.csv", index=False, encoding="utf-8-sig")
final_summary.to_csv(eval_dir / "repair_comparison.csv", index=False, encoding="utf-8-sig")
error_summary.to_csv(eval_dir / "repair_error_summary.csv", index=False, encoding="utf-8-sig")

print("Số bản ghi:", len(full))
print("Số SQL được sửa:", len(repair_eval))
print("Đã lưu kết quả tại:", eval_dir)

display(final_summary)

Số bản ghi: 344
Số SQL được sửa: 284
Đã lưu kết quả tại: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\related_methods\ontology_guided_repair\evaluation


,Model,Method,Questions,Baseline_Correct,Correct_After,EA Before (%),EA After (%),Delta EA (pp)
0,NSQL-350M,ontology_guided,43,3,3,6.98,6.98,0.00
1,NSQL-350M,schema_only,43,3,3,6.98,6.98,0.00
2,NSQL-Llama-2-7B,ontology_guided,43,0,3,0.00,6.98,6.98
3,NSQL-Llama-2-7B,schema_only,43,0,4,0.00,9.30,9.30
4,Qwen2.5-Coder-3B,ontology_guided,43,4,4,9.30,9.30,0.00
5,Qwen2.5-Coder-3B,schema_only,43,4,4,9.30,9.30,0.00
6,SQLCoder-7B-2,ontology_guided,43,4,4,9.30,9.30,0.00
7,SQLCoder-7B-2,schema_only,43,4,4,9.30,9.30,0.00
